In [ ]:
import json
import re
import numpy as np
from pathlib import Path
import pandas as pd
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'text.usetex': True,
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
})

Path('plots').mkdir(exist_ok=True)
R_lb = 1


In [ ]:
CRITERION = Path('../target/criterion')

def parse_int(s, key):
    m = re.search(rf'{key}=(\d+)', s)
    return int(m.group(1)) if m else None

rows = []
for f in CRITERION.rglob('new/estimates.json'):
    path_str = str(f.relative_to(CRITERION))
    cycles = json.loads(f.read_text())['mean']['point_estimate']
    if path_str.startswith('mklhs_keygen'):
        scheme, op, R, t = 'mklhs', 'keygen', None, None
    elif path_str.startswith('mklhs_sign'):
        scheme, op, R, t = 'mklhs', 'sign', None, None
    elif path_str.startswith('mklhs'):
        scheme, op = 'mklhs', 'eval' if 'eval' in path_str else 'verify'
        R, t = None, parse_int(path_str, 't')
    elif path_str.startswith('mkqhs_cbr_msq_keygen'):
        scheme, op, R, t = 'mkqhs', 'keygen', None, None
    elif path_str.startswith('mkqhs_cbr_msq_sign'):
        scheme, op, R, t = 'mkqhs', 'sign', None, None
    else:
        scheme, op = 'mkqhs', 'eval' if 'eval' in path_str else 'verify'
        R, t = parse_int(path_str, 'R'), parse_int(path_str, 't')
    rows.append({'scheme': scheme, 'op': op, 'R': R, 't': t, 'cycles': cycles})

df = pd.DataFrame(rows)
df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, op in zip(axes, ['eval', 'verify']):
    sub = df[(df['scheme'] == 'mklhs') & (df['op'] == op)].dropna(subset=['t'])
    m = LinearRegression().fit(sub[['t']], sub['cycles'])
    r2 = m.score(sub[['t']], sub['cycles'])

    t_line = np.linspace(sub['t'].min(), sub['t'].max(), 100)
    ax.plot(t_line, m.intercept_ + m.coef_[0] * t_line, color='steelblue', label='fit')
    ax.scatter(sub['t'], sub['cycles'], color='red', zorder=5)
    ax.set_xlabel(r'$t$ (signers)')
    ax.set_ylabel(r'cycles')
    ax.set_title(
        r'$\mathsf{mklhs}$' + r'$\mathsf{.' + op.capitalize() + '}$' + '\n'
        r'$\approx ' + f'{m.intercept_/1e6:.2f}' + r'\mathrm{M} + ' + f'{m.coef_[0]/1e6:.2f}' + r'\mathrm{M} \cdot t$'
        + f'\n$R^2 = {r2:.4f}$'
    )

fig.suptitle(r'$\mathsf{mklhs}$ cycle counts', fontsize=14)
plt.tight_layout()
plt.savefig('plots/mklhs_cycles.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# mklhs vs mkqhs at R=0
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, op in zip(axes, ['eval', 'verify']):
    lhs = df[(df['scheme'] == 'mklhs') & (df['op'] == op)].dropna(subset=['t'])
    qhs = df[(df['scheme'] == 'mkqhs') & (df['op'] == op) & (df['R'] == 0)].dropna(subset=['t'])

    m_lhs = LinearRegression().fit(lhs[['t']], lhs['cycles'])
    m_qhs = LinearRegression().fit(qhs[['t']], qhs['cycles'])
    r2_lhs = m_lhs.score(lhs[['t']], lhs['cycles'])
    r2_qhs = m_qhs.score(qhs[['t']], qhs['cycles'])

    t_line = np.linspace(
        min(lhs['t'].min(), qhs['t'].min()),
        max(lhs['t'].max(), qhs['t'].max()),
        100,
    )

    ax.plot(t_line, m_lhs.intercept_ + m_lhs.coef_[0] * t_line,
            color='steelblue', label=r'$\mathsf{mklhs}$' + f' ($R^2={r2_lhs:.4f}$)')
    ax.scatter(lhs['t'], lhs['cycles'], color='steelblue', zorder=5)

    ax.plot(t_line, m_qhs.intercept_ + m_qhs.coef_[0] * t_line,
            color='tomato', label=r'$\mathsf{mkqhs}$, $R=0$' + f' ($R^2={r2_qhs:.4f}$)')
    ax.scatter(qhs['t'], qhs['cycles'], color='tomato', zorder=5)

    ax.set_xlabel(r'$t$ (signers)')
    ax.set_ylabel(r'cycles')
    ax.set_title(
        r'\textsc{' + r'$\mathsf{' + op.capitalize() + '}$' + r'}: $\mathsf{mklhs}$ vs $\mathsf{mkqhs}$ at $R=0$' + '\n'
        r'$\mathsf{mklhs} \approx ' + f'{m_lhs.coef_[0]/1e6:.2f}' + r'\mathrm{M}\cdot t'
        + (f' {m_lhs.intercept_/1e6:+.2f}' + r'\mathrm{M}$') + '\n'
        r'$\mathsf{mkqhs} \approx ' + f'{m_qhs.coef_[0]/1e6:.2f}' + r'\mathrm{M}\cdot t'
        + (f' {m_qhs.intercept_/1e6:+.2f}' + r'\mathrm{M}$')
    )
    ax.legend()

fig.suptitle(r'$\mathsf{mklhs}$ vs $\mathsf{mkqhs}$ at $R=0$', fontsize=14)
plt.tight_layout()
plt.savefig('plots/comparison_R0.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# mkqhs: R + t model (no interaction), R > 0
fig = plt.figure(figsize=(12, 5))

for idx, op in enumerate(['eval', 'verify'], 1):
    sub = df[(df['scheme'] == 'mkqhs') & (df['op'] == op)].dropna(subset=['R', 't'])
    sub = sub[sub['R'] >= R_lb].copy()
    m = LinearRegression().fit(sub[['R', 't']], sub['cycles'])
    r2 = m.score(sub[['R', 't']], sub['cycles'])

    ax = fig.add_subplot(1, 2, idx, projection='3d')
    R_g, t_g = np.meshgrid(
        np.linspace(sub['R'].min(), sub['R'].max(), 30),
        np.linspace(sub['t'].min(), sub['t'].max(), 30),
    )
    z_g = m.intercept_ + m.coef_[0] * R_g + m.coef_[1] * t_g
    ax.plot_surface(R_g, t_g, z_g, alpha=0.3, color='steelblue')
    ax.scatter(sub['R'], sub['t'], sub['cycles'], color='red', zorder=5)
    ax.set_xlabel(r'$R$')
    ax.set_ylabel(r'$t$')
    ax.set_zlabel(r'cycles')
    ax.set_title(
        r'$\mathsf{mkqhs}$' + r'$\mathsf{.' + op.capitalize() + '}$' + r' ($R + t$ model)' + '\n'
        r'$\approx ' + f'{m.coef_[0]/1e6:.2f}' + r'\mathrm{M}\cdot R + ' + f'{m.coef_[1]/1e6:.2f}' + r'\mathrm{M}\cdot t$'
        + f'\n$R^2 = {r2:.4f}$'
    )

fig.suptitle(r'$\mathsf{mkqhs}$ cycle counts: $R + t$ model', fontsize=14)
plt.tight_layout()
plt.savefig('plots/mkqhs_cycles_R_plus_t.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# mkqhs: R*t interaction model, R > 0
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, op in zip(axes, ['eval', 'verify']):
    sub = df[(df['scheme'] == 'mkqhs') & (df['op'] == op)].dropna(subset=['R', 't'])
    sub = sub[sub['R'] >= R_lb].copy()
    sub['Rt'] = sub['R'] * sub['t']
    m = LinearRegression().fit(sub[['R', 't', 'Rt']], sub['cycles'])
    r2 = m.score(sub[['R', 't', 'Rt']], sub['cycles'])

    r_vals = sorted(sub['R'].unique())
    colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(r_vals)))
    t_line = np.linspace(sub['t'].min(), sub['t'].max(), 100)

    for color, r_val in zip(colors, r_vals):
        grp = sub[sub['R'] == r_val]
        intercept_r = m.intercept_ + m.coef_[0] * r_val
        slope_r = m.coef_[1] + m.coef_[2] * r_val
        ax.plot(t_line, intercept_r + slope_r * t_line, color=color, label=f'$R={int(r_val)}$')
        ax.scatter(grp['t'], grp['cycles'], color=color, zorder=5)

    ax.set_xlabel(r'$t$ (signers)')
    ax.set_ylabel(r'cycles')
    ax.set_title(
        r'$\mathsf{mkqhs}$' + r'$\mathsf{.' + op.capitalize() + '}$' + r' ($R \cdot t$ interaction)' + '\n'
        r'$\approx ' + f'{m.coef_[0]/1e6:.2f}' + r'\mathrm{M}\cdot R + ' + f'{m.coef_[1]/1e6:.2f}' + r'\mathrm{M}\cdot t + ' + f'{m.coef_[2]/1e6:.2f}' + r'\mathrm{M}\cdot Rt$'
        + f'\n$R^2 = {r2:.4f}$'
    )
    ax.legend()

fig.suptitle(r'$\mathsf{mkqhs}$ cycle counts: $R \cdot t$ interaction model', fontsize=14)
plt.tight_layout()
plt.savefig('plots/mkqhs_cycles_R_times_t.pdf', bbox_inches='tight')
plt.show()